# Hotel Booking Demand — Pipeline, Split and Metric Harness

**Module:** IT3091 Machine Learning · **Group:** 2026-AI-46 · **Lab group:** Y3.S1.WD.AI.0101
**Stage 2 of 3** — completing the *frozen shared foundation*. Owner: project lead.
Runs after `01_eda_and_cleaning.ipynb`, before any `03_*` model notebook.

**Business question.** A hotel chain's revenue and operations manager needs to know, *before arrival*,
which bookings are likely to be cancelled. Notebook 01 turned the raw data into a cleaned,
booking-time-safe table and a frozen feature contract. This notebook builds the machinery that every
model in Stage 3 will share, so that the final model comparison is fair.

**What this notebook produces**

| Output | Used by |
| --- | --- |
| `data/processed/train.csv` — the 80% training rows, frozen | every `03_*` model notebook |
| `data/processed/test.csv` — the 20% hold-out, **not opened until the single final scoring pass** | Stage 3, once |
| `data/processed/split_manifest.json` — seed, sizes, class rates, input fingerprint, CV and feature groups | reproducibility check, report |
| `reports/training_foundation.md` — the "how to add your model" handover | M2–M4 |
| A `ColumnTransformer`, a `StratifiedKFold` definition and a metric harness (defined here, copied by the model notebooks) | Stage 3 |

**What this notebook deliberately does *not* do.** It does not fit, tune or score any candidate model,
and it never reads `test.csv` back. It stops exactly at the point where four members can work in
parallel on an identical starting line.

**Reference notebooks (plan.md §16).** All five published Kaggle notebooks on this dataset were read on
2026-09-03 as a structural benchmark. Every one of them **fits its encoders and scaler on the whole
dataset and then splits** (departure **D5** in plan.md §16). That leaks the test set into the training
statistics and inflates every score they report. The single most important design choice in this
notebook is the opposite: **every transform is fitted inside the training folds only.**

---
## 1. Setup

In [1]:
import hashlib
import json
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.dummy import DummyClassifier
from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    average_precision_score, roc_auc_score, f1_score, recall_score,
    precision_score, accuracy_score, confusion_matrix, brier_score_loss,
    make_scorer,
)

sns.set_theme(style="whitegrid")
PRIMARY, ACCENT = "#2B5C8F", "#E05A47"
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.titleweight"] = "bold"
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 150)

RANDOM_SEED = 42          # identical to notebook 01
TEST_SIZE = 0.20
N_SPLITS = 5
TARGET = "is_canceled"
np.random.seed(RANDOM_SEED)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = ROOT / "data" / "processed"
REPORTS_DIR = ROOT / "reports"
CLEAN_CSV = PROCESSED_DIR / "hotel_bookings_clean.csv"
CONTRACT_CSV = REPORTS_DIR / "feature_roles.csv"

print("sklearn", sklearn.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)
print("seed", RANDOM_SEED, "| test_size", TEST_SIZE, "| cv folds", N_SPLITS)

sklearn 1.9.0 | pandas 3.0.5 | numpy 2.4.6
seed 42 | test_size 0.2 | cv folds 5


---
## 2. Load the cleaned table and the frozen feature contract

Notebook 01 wrote two files this notebook depends on and **nothing else**:

- `data/processed/hotel_bookings_clean.csv` — 86,965 cleaned rows, booking-time-safe.
- `reports/feature_roles.csv` — the frozen contract: a `role` for every column
  (`feature` / `target` / `post-booking` / `helper`).

We record the SHA-256 of the cleaned file so the split can always be traced back to the exact input
that produced it.

In [2]:
def sha256(path):
    h = hashlib.sha256()
    h.update(Path(path).read_bytes())
    return h.hexdigest()

clean = pd.read_csv(CLEAN_CSV)
contract = pd.read_csv(CONTRACT_CSV)
CLEAN_SHA = sha256(CLEAN_CSV)

print(f"clean    : {clean.shape[0]:,} rows x {clean.shape[1]} columns")
print(f"contract : {contract.shape[0]} columns described")
print(f"sha256   : {CLEAN_SHA}")
contract

clean    : 86,965 rows x 43 columns
contract : 43 columns described
sha256   : 51bb864402acaef61fe6175ffec1afea33f7c8173ab08bfff03573d5f686b17b


,column,dtype,n_unique,role
0,adr,float64,8864,feature
1,adr_per_guest,float64,10800,feature
2,adults,int64,14,feature
3,agent,str,334,feature
4,arrival_date_day_of_month,int64,31,feature
5,arrival_date_month,category,12,feature
6,arrival_date_week_number,int64,53,feature
7,arrival_date_year,int64,3,feature
8,arrival_season,str,4,feature
9,babies,int64,5,feature


---
## 3. Enforce the feature contract

The contract is the frozen agreement from notebook 01 §11. This notebook **reads it and never
re-argues it**. Three things happen here:

1. **Select `X` and `y` by role.** Only the 36 columns marked `feature` become model inputs. `target`
   is `y`. Everything marked `post-booking` (`assigned_room_type`, `booking_changes`,
   `days_in_waiting_list`, `required_car_parking_spaces`, `room_changed`) or `helper` (`arrival_date`)
   is dropped — those are known only at or after check-in and would leak the outcome (notebook 01
   findings L2–L3).
2. **Restore categorical dtypes.** A CSV round-trip loses them: `agent` in particular comes back as
   `int64` and must be a string category again, exactly as the contract says (`agent, str, 334`).
3. **Split the features into three preprocessing groups** — see the table below.

In [3]:
role_of = dict(zip(contract["column"], contract["role"]))
contract_dtype = dict(zip(contract["column"], contract["dtype"]))

FEATURES = [c for c in contract.loc[contract["role"] == "feature", "column"]]
DROP_COLS = [c for c, r in role_of.items() if r in {"post-booking", "helper"}]
LEAKAGE_COLS = ["reservation_status", "reservation_status_date"]   # dropped already in notebook 01

# categorical = the contract says str / category; everything else is numeric
CATEGORICAL = [c for c in FEATURES if contract_dtype[c] in {"str", "category", "object"}]
NUMERIC = [c for c in FEATURES if c not in CATEGORICAL]

# high-cardinality categoricals get a different encoder (see §5)
HIGH_CARD = ["country", "agent"]
LOW_CARD = [c for c in CATEGORICAL if c not in HIGH_CARD]

# restore the categorical dtype the CSV lost
for c in CATEGORICAL:
    clean[c] = clean[c].astype("string").astype(str)

X = clean[FEATURES].copy()
y = clean[TARGET].astype(int)

print(f"model features        : {len(FEATURES)}")
print(f"  numeric  -> scale    : {len(NUMERIC)}")
print(f"  low-card -> one-hot  : {len(LOW_CARD)}  {LOW_CARD}")
print(f"  high-card-> frequency: {len(HIGH_CARD)}  " +
      ", ".join(f"{c} ({clean[c].nunique()} levels)" for c in HIGH_CARD))
print(f"dropped (post-booking/helper): {sorted(DROP_COLS)}")

model features        : 36
  numeric  -> scale    : 24
  low-card -> one-hot  : 10  ['arrival_date_month', 'arrival_season', 'customer_type', 'deposit_type', 'distribution_channel', 'hotel', 'lead_time_group', 'market_segment', 'meal', 'reserved_room_type']
  high-card-> frequency: 2  country (178 levels), agent (334 levels)
dropped (post-booking/helper): ['arrival_date', 'assigned_room_type', 'booking_changes', 'days_in_waiting_list', 'required_car_parking_spaces', 'room_changed']


**Why these three groups.**

| Group | Columns | Treatment | Why |
| --- | --- | --- | --- |
| Numeric | 24 (lead time, adr, party size, guest history, date parts, flags…) | `StandardScaler` | Logistic Regression and any distance-based model need comparable scales; the heavy `lead_time` / `adr` tails (notebook 01 §5.4) are handled here rather than by deleting rows. Scaling is a harmless no-op for the tree models, and one shared transformer keeps every model's input **identical**, which is what makes the comparison valid. |
| Low-cardinality categorical | 10 (`hotel`, `meal`, `market_segment`, `deposit_type`, `customer_type`, `reserved_room_type`, `arrival_date_month`, `arrival_season`, `lead_time_group`, `distribution_channel`) | `OneHotEncoder(handle_unknown="ignore")` | ≤ 12 levels each — one-hot is transparent and cheap (~57 columns total). `handle_unknown="ignore"` means a category not seen in a given training fold becomes an all-zero row instead of an error. |
| High-cardinality categorical | `country` (178), `agent` (334) | `FrequencyEncoder` (§5) | One-hot would add ~510 sparse columns, most of them near-empty — it inflates dimensionality, slows every model and hurts the linear baseline. Frequency encoding gives **one** dense column each. It is fitted per fold and never looks at `y`, so it cannot leak the target. |

`no imputation step is included`: notebook 01's guard rails already assert zero missing values, and
plan.md §6 records that every missing-value case was settled on its own evidence during cleaning
(`company`/`agent` → flags, `country` → explicit `Unknown`, `children` → 0). Adding a `SimpleImputer`
now would be dead code that hides that decision.

---
## 4. The held-out test set

**Split first, before anything else touches the data.** This is step 1 of the evaluation protocol in
plan.md §8. The 20% test set is put aside now and is not read again in this notebook, or in any model
notebook, until the *single* final scoring pass in Stage 3. Every tuning decision is made on the
training rows only.

- **Stratified** on `is_canceled` so the 27.3% cancellation rate is identical in both parts — otherwise
  the test score would partly measure a class-balance accident.
- **`random_state=42`**, the same seed as notebook 01, so the split is byte-for-byte reproducible.
- **80 / 20**: with ~87k rows, 20% is ~17.4k test bookings — a large enough hold-out to estimate
  recall on the minority (cancelled) class precisely, while leaving ~70k for training and CV.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y,
)

split_tbl = pd.DataFrame({
    "rows": [len(X_train), len(X_test), len(X)],
    "share": [len(X_train) / len(X), len(X_test) / len(X), 1.0],
    "cancelled_rate": [y_train.mean(), y_test.mean(), y.mean()],
}, index=["train", "test (held out)", "all"])
split_tbl.round(4)

,rows,share,cancelled_rate
train,69572,0.8,0.2731
test (held out),17393,0.2,0.2731
all,86965,1.0,0.2731


**Reading.** The cancellation rate matches to four decimal places across train, test and the full
table, so the hold-out is a faithful miniature of the data. From here on, `X_test` / `y_test` are
treated as if they do not exist.

---
## 5. Leakage-safe preprocessing

### 5.1 The rule

Every transform below is wrapped in a `Pipeline` and fitted **only on the training portion of each
cross-validation fold**. Nothing is `.fit()` on the full training set before cross-validation, and
certainly not on `X` — that is departure **D5** from the five reference notebooks (plan.md §16). If a
scaler learns its mean from rows that later sit in the validation fold, the reported score is
optimistic by an amount you cannot measure after the fact.

### 5.2 A frequency encoder for `country` and `agent`

`country` has 178 levels and `agent` has 334. One-hot encoding them would add ~510 mostly-sparse
columns. Instead we map each category to **how often it appears in the fold's training rows** — a
single dense number per column. It is a `TransformerMixin`, so scikit-learn fits it inside the folds
like any other step. It never receives `y`, so unlike target encoding it has *no* route to leak the
label. Categories that appear only in the validation fold (or later in the test set) encode to `0.0`,
which is the correct "vanishingly rare here" signal.

In [5]:
class FrequencyEncoder(BaseEstimator, TransformerMixin):
    """Encode each categorical column by its training-set relative frequency.

    Fitted inside the pipeline, so within a CV fold it only sees that fold's
    training rows. Unseen categories -> 0.0. Does not use ``y``: no target leakage.
    """

    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        self.columns_ = list(X.columns)
        self.freq_ = {
            c: X[c].astype(str).value_counts(normalize=True).to_dict()
            for c in self.columns_
        }
        return self

    def transform(self, X):
        X = pd.DataFrame(X)
        out = np.zeros((len(X), len(self.columns_)), dtype=float)
        for j, c in enumerate(self.columns_):
            out[:, j] = X[c].astype(str).map(self.freq_[c]).fillna(0.0).to_numpy()
        return out

    def get_feature_names_out(self, input_features=None):
        return np.asarray([f"{c}_freq" for c in self.columns_], dtype=object)

### 5.3 The one `ColumnTransformer` every model shares

In [6]:
preprocess = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), NUMERIC),
        ("onehot", OneHotEncoder(handle_unknown="ignore"), LOW_CARD),
        ("frequency", FrequencyEncoder(), HIGH_CARD),
    ],
    remainder="drop",
    verbose_feature_names_out=True,
)

def make_pipeline(estimator):
    """Wrap a classifier so preprocessing is refit on every training fold."""
    return Pipeline([("preprocess", preprocess), ("model", estimator)])

# Inspect the shape it produces — fit on the TRAINING data only, for display.
preview = preprocess.fit(X_train, y_train)
feat_out = preview.get_feature_names_out()
print(f"columns into the model : {len(feat_out)}")
groups = pd.Series([n.split("__")[0] for n in feat_out]).value_counts()
print(groups.to_string())

columns into the model : 83
onehot       57
numeric      24
frequency     2


**Reading.** 83 columns reach the estimator: 24 scaled numerics, 57 one-hot dummies and 2 frequency
columns. `preprocess` is shown fitted here only to report that shape — in Stage 3 it is always used
*inside* `make_pipeline(...)` so it is refit per fold.

---
## 6. Cross-validation protocol

One fold definition, frozen and shared. `StratifiedKFold(n_splits=5, shuffle=True,
random_state=42)`:

- **5 folds** — a standard bias/variance trade-off; each fold trains on ~56k rows and validates on
  ~14k, enough to score minority-class recall stably.
- **Stratified** — every fold holds the 27.3% cancellation rate, for the same reason as the test
  split.
- **`shuffle=True` with a fixed seed** — the cleaned file is not in random order (it is de-duplicated
  and roughly time-ordered), so shuffling avoids folds that are accidentally a single season.
- Defined on `X_train` only. The hold-out never enters a fold.

In [7]:
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_SEED)

fold_rows = []
for i, (tr, va) in enumerate(cv.split(X_train, y_train), start=1):
    fold_rows.append({"fold": i, "train_rows": len(tr), "val_rows": len(va),
                      "val_cancelled_rate": round(y_train.iloc[va].mean(), 4)})
pd.DataFrame(fold_rows).set_index("fold")

,train_rows,val_rows,val_cancelled_rate
fold,,,
1,55657,13915,0.2731
2,55657,13915,0.2731
3,55658,13914,0.2731
4,55658,13914,0.2731
5,55658,13914,0.2730


---
## 7. The metric harness

Notebook 01 §13 and R1's negative result (a Random Forest at 72.8% accuracy that recalls **57 of
4,805** cancellations) set the rule: **accuracy alone is worthless here**. Predicting "never cancels"
already scores ~72.7%.

| Tier | Metric | Why |
| --- | --- | --- |
| **Primary** | PR-AUC (average precision) | threshold-free, focused on the positive (cancelled) class under imbalance |
| **Primary** | Recall (cancelled) | a missed cancellation is the expensive error for a revenue manager |
| **Primary** | F1 (cancelled) | balances recall against false alarms at the operating threshold |
| Secondary | ROC-AUC | ranking quality, comparable to published results |
| Secondary | Precision, accuracy | context, not decision criteria |
| Secondary | Confusion matrix | the raw error counts a manager can reason about |
| Secondary | Calibration (Brier + curve) | whether the probability can be read as a probability, needed for threshold-setting |

`SCORERS` plugs straight into `cross_validate`. `evaluate()` scores a single set of predictions at a
chosen threshold. Both draw from **one** dict of metric functions, so there is a single source of
truth.

In [8]:
# one source of truth for the probability-based metrics
_PROBA_METRICS = {"pr_auc": average_precision_score, "roc_auc": roc_auc_score}

SCORERS = {name: make_scorer(fn, response_method="predict_proba")
           for name, fn in _PROBA_METRICS.items()}
SCORERS.update({
    "f1": make_scorer(f1_score, zero_division=0),
    "recall": make_scorer(recall_score, zero_division=0),
    "precision": make_scorer(precision_score, zero_division=0),
    "accuracy": make_scorer(accuracy_score),
})


def evaluate(y_true, y_prob, threshold=0.5):
    """Score one set of predicted probabilities at a given decision threshold."""
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    y_hat = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_hat, labels=[0, 1]).ravel()
    return {
        "threshold": threshold,
        "pr_auc": average_precision_score(y_true, y_prob),
        "roc_auc": roc_auc_score(y_true, y_prob),
        "f1": f1_score(y_true, y_hat, zero_division=0),
        "recall": recall_score(y_true, y_hat, zero_division=0),
        "precision": precision_score(y_true, y_hat, zero_division=0),
        "accuracy": accuracy_score(y_true, y_hat),
        "brier": brier_score_loss(y_true, y_prob),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }


def calibration_table(y_true, y_prob, n_bins=10):
    """Reliability curve points plus the Brier score."""
    frac_pos, mean_pred = calibration_curve(y_true, y_prob, n_bins=n_bins, strategy="quantile")
    return pd.DataFrame({"mean_predicted": mean_pred, "fraction_positive": frac_pos}), \
        brier_score_loss(y_true, y_prob)


def cv_report(estimator, X_tr=None, y_tr=None):
    """Run the shared CV on a pipeline and return mean +/- std for every scorer."""
    X_tr = X_train if X_tr is None else X_tr
    y_tr = y_train if y_tr is None else y_tr
    res = cross_validate(estimator, X_tr, y_tr, cv=cv, scoring=SCORERS, n_jobs=None)
    return pd.DataFrame({
        "mean": {k: res[f"test_{k}"].mean() for k in SCORERS},
        "std": {k: res[f"test_{k}"].std() for k in SCORERS},
    })

### 7.1 Reference floor — the scores every model in notebook 03 must beat

This is **not** a candidate model. `DummyClassifier(strategy="prior")` always predicts the training
cancellation rate. Running it through the real harness does two things: it proves the harness works end
to end, and it prints the no-skill baseline that plan.md §8 says every model must be judged against.

In [9]:
baseline = make_pipeline(DummyClassifier(strategy="prior"))
baseline_cv = cv_report(baseline)
print("DummyClassifier(prior) -- 5-fold CV on the training data")
print(baseline_cv.round(4).to_string())

# and the same metrics as a single-prediction check via evaluate()
prior_p = np.full(len(y_train), y_train.mean())
print("\nevaluate() on the constant prior probability (threshold 0.5):")
for k, v in evaluate(y_train, prior_p).items():
    print(f"  {k:>10}: {v:.4f}" if isinstance(v, float) else f"  {k:>10}: {v}")

DummyClassifier(prior) -- 5-fold CV on the training data
             mean  std
pr_auc     0.2731  0.0
roc_auc    0.5000  0.0
f1         0.0000  0.0
recall     0.0000  0.0
precision  0.0000  0.0
accuracy   0.7269  0.0

evaluate() on the constant prior probability (threshold 0.5):
   threshold: 0.5000
      pr_auc: 0.2731
     roc_auc: 0.5000
          f1: 0.0000
      recall: 0.0000
   precision: 0.0000
    accuracy: 0.7269
       brier: 0.1985
          tn: 50573
          fp: 0
          fn: 18999
          tp: 0


**Reading.** PR-AUC sits at the positive rate (~0.27) and ROC-AUC at 0.50 — exactly a model with no
information. Recall is 0 at threshold 0.5 (the prior probability is below 0.5, so nothing is flagged).
Any real model has to clear this bar on the primary metrics, not on accuracy.

---
## 8. Guard rails

Notebook 03 must not start if any of these fail.

In [10]:
# 8.1 no outcome-revealing column can reach the model
POST_BOOKING = [c for c, r in role_of.items() if r == "post-booking"]
HELPER = [c for c, r in role_of.items() if r == "helper"]
assert not (set(LEAKAGE_COLS) & set(X.columns)), "a leakage column is in X"
assert not (set(POST_BOOKING) & set(X.columns)), "a post-booking column is in X"
assert not (set(HELPER) & set(X.columns)), "a helper column is in X"
assert TARGET not in X.columns, "target leaked into X"
assert list(X.columns) == FEATURES and len(FEATURES) == 36, "feature set drifted from the contract"

# 8.2 the hold-out is disjoint from training and from every CV fold
assert set(X_train.index).isdisjoint(X_test.index), "train and test rows overlap"
assert len(X_train) + len(X_test) == len(X), "rows lost or duplicated in the split"
for tr, va in cv.split(X_train, y_train):
    fold_val_idx = set(X_train.iloc[va].index)
    assert fold_val_idx.isdisjoint(X_test.index), "a CV fold contains hold-out rows"

# 8.3 stratification really held
assert abs(y_train.mean() - y.mean()) < 5e-3, "train class balance drifted"
assert abs(y_test.mean() - y.mean()) < 5e-3, "test class balance drifted"

# 8.4 the split is deterministic — re-run it and the row ids match exactly
X_tr2, X_te2, _, _ = train_test_split(X, y, test_size=TEST_SIZE,
                                      random_state=RANDOM_SEED, stratify=y)
assert X_tr2.index.equals(X_train.index) and X_te2.index.equals(X_test.index), "split not reproducible"

# 8.5 preprocessing carries nothing that is not one of the contract features
assert set(NUMERIC) | set(LOW_CARD) | set(HIGH_CARD) == set(FEATURES), "a feature is unassigned"
assert not (set(NUMERIC) & set(CATEGORICAL)), "a column is both numeric and categorical"

print("all guard rails passed")

all guard rails passed

---
## 9. Persist the frozen foundation

`train.csv` and `test.csv` are written with the cleaned-table row index as the first column, so any
later notebook can prove its rows came from this split and that the two files never overlap. The
manifest is the audit record: the exact input fingerprint, the seed, the sizes, the class rates, the
CV definition and the feature groups. These files are rebuildable and stay untracked
(`data/processed/`), consistent with plan.md §12.

In [11]:
train_df = clean.loc[X_train.index].copy()
test_df = clean.loc[X_test.index].copy()

TRAIN_CSV = PROCESSED_DIR / "train.csv"
TEST_CSV = PROCESSED_DIR / "test.csv"
MANIFEST = PROCESSED_DIR / "split_manifest.json"

train_df.to_csv(TRAIN_CSV, index=True, index_label="row_id")
test_df.to_csv(TEST_CSV, index=True, index_label="row_id")

manifest = {
    "created_by": "notebooks/02_pipeline_and_split.ipynb",
    "python": platform.python_version(),
    "sklearn": sklearn.__version__,
    "random_seed": RANDOM_SEED,
    "input": {
        "path": "data/processed/hotel_bookings_clean.csv",
        "sha256": CLEAN_SHA,
        "rows": int(len(clean)),
    },
    "split": {
        "method": "sklearn.model_selection.train_test_split",
        "test_size": TEST_SIZE,
        "stratify": TARGET,
        "train_rows": int(len(train_df)),
        "test_rows": int(len(test_df)),
        "train_cancelled_rate": round(float(y_train.mean()), 6),
        "test_cancelled_rate": round(float(y_test.mean()), 6),
    },
    "cross_validation": {
        "class": "StratifiedKFold",
        "n_splits": N_SPLITS,
        "shuffle": True,
        "random_state": RANDOM_SEED,
        "fitted_on": "train.csv only",
    },
    "target": TARGET,
    "feature_groups": {
        "numeric_scaled": NUMERIC,
        "categorical_onehot": LOW_CARD,
        "categorical_frequency": HIGH_CARD,
    },
    "excluded": {
        "leakage": LEAKAGE_COLS,
        "post_booking": sorted(POST_BOOKING),
        "helper": sorted(HELPER),
    },
    "primary_metrics": ["pr_auc", "recall", "f1"],
    "secondary_metrics": ["roc_auc", "precision", "accuracy", "brier", "confusion_matrix"],
    "artifacts": {
        "train_csv": "data/processed/train.csv",
        "test_csv": "data/processed/test.csv",
    },
}

MANIFEST.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
manifest["artifacts"]["train_csv_sha256"] = sha256(TRAIN_CSV)
manifest["artifacts"]["test_csv_sha256"] = sha256(TEST_CSV)
MANIFEST.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

for p in (TRAIN_CSV, TEST_CSV, MANIFEST):
    print(f"wrote {p.relative_to(ROOT)}  ({p.stat().st_size/1e6:.2f} MB)")

wrote data\processed\train.csv  (12.33 MB)
wrote data\processed\test.csv  (3.08 MB)
wrote data\processed\split_manifest.json  (0.00 MB)


---
## 10. Handover to the model notebooks

Section 12 below writes `reports/training_foundation.md`. The core of it is the snippet every `03_*`
notebook starts with — it loads **only the training data**, so there is no way to accidentally look at
the hold-out while tuning:

```python
import json, pandas as pd
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
M = json.loads((ROOT / "data/processed/split_manifest.json").read_text())

train = pd.read_csv(ROOT / M["artifacts"]["train_csv"], index_col="row_id")   # training rows only

num  = M["feature_groups"]["numeric_scaled"]
ohe  = M["feature_groups"]["categorical_onehot"]
freq = M["feature_groups"]["categorical_frequency"]
for c in ohe + freq:
    train[c] = train[c].astype(str)

X_train, y_train = train[num + ohe + freq], train[M["target"]].astype(int)
```

`test.csv` is **not loaded here**. It is read exactly once, in the Stage 3 scoring notebook, only after
the recommended model and its operating threshold are locked:

```python
# Stage 3 only — after the model is chosen. Never in a model-tuning notebook.
test = pd.read_csv(ROOT / M["artifacts"]["test_csv"], index_col="row_id")
for c in ohe + freq:
    test[c] = test[c].astype(str)
X_test, y_test = test[num + ohe + freq], test[M["target"]].astype(int)
```

Then copy the `FrequencyEncoder`, `preprocess`, `make_pipeline`, `cv`, `SCORERS`, `evaluate` and
`cv_report` definitions from this notebook (§5–§7) and train exactly one model inside `cv`.

---
## 11. A note on what is *not* here

No model is fitted or tuned (the `DummyClassifier` in §7.1 is a reference floor, not a candidate), and
`test.csv` is never read back. Hyperparameter search, per-model calibration, feature importance and the
final single test-set pass all belong to Stage 3, one model per member, each on the frozen foundation
written above.

---
## 12. Write the handover document

In [12]:
foundation_md = f"""# Training foundation — Hotel Booking Demand

Group 2026-AI-46 · IT3091 Machine Learning
Generated by `notebooks/02_pipeline_and_split.ipynb`. Do not edit by hand — re-run the notebook.

This is the **frozen shared foundation**. Every `03_*` model notebook starts from exactly these files,
this split, this preprocessing and this metric set, so the final model comparison is fair.

## Inputs (and nothing else)

| File | SHA-256 | Rows |
| --- | --- | --- |
| `data/processed/hotel_bookings_clean.csv` | `{CLEAN_SHA}` | {len(clean):,} |
| `reports/feature_roles.csv` | frozen contract from notebook 01 | {len(contract)} columns |

## The split

| | Rows | Share | Cancelled rate |
| --- | ---: | ---: | ---: |
| Train (`data/processed/train.csv`) | {len(train_df):,} | {len(train_df)/len(clean):.1%} | {y_train.mean():.4f} |
| Test  (`data/processed/test.csv`)  | {len(test_df):,} | {len(test_df)/len(clean):.1%} | {y_test.mean():.4f} |

- `sklearn.model_selection.train_test_split`, `test_size={TEST_SIZE}`, `random_state={RANDOM_SEED}`,
  `stratify=is_canceled`.
- Written with the cleaned-table row index as `row_id` (first column). Train and test `row_id` sets are
  disjoint and cover every cleaned row.
- **`test.csv` is not opened until the single final scoring pass in Stage 3.**

## Cross-validation (training data only)

`StratifiedKFold(n_splits={N_SPLITS}, shuffle=True, random_state={RANDOM_SEED})` — shared by every model.
All preprocessing is fitted **inside** these folds (departure D5 from the reference notebooks,
plan.md §16): fitting a scaler or an encoder on rows that later sit in the validation fold inflates the
score by an unmeasurable amount.

## Feature groups ({len(FEATURES)} features)

| Group | Count | Transform | Columns |
| --- | ---: | --- | --- |
| Numeric | {len(NUMERIC)} | `StandardScaler` | {", ".join(NUMERIC)} |
| Low-cardinality categorical | {len(LOW_CARD)} | `OneHotEncoder(handle_unknown="ignore")` | {", ".join(LOW_CARD)} |
| High-cardinality categorical | {len(HIGH_CARD)} | `FrequencyEncoder` (fold-local, ignores `y`) | {", ".join(HIGH_CARD)} |

No imputation step: notebook 01 guarantees zero missing values (plan.md §6).
One `ColumnTransformer` → **{len(feat_out)}** columns into the estimator ({groups.get("numeric",0)} scaled,
{groups.get("onehot",0)} one-hot, {groups.get("frequency",0)} frequency).

### Excluded, and why

- **Leakage** — {", ".join("`"+c+"`" for c in LEAKAGE_COLS)}: set at/after the outcome (notebook 01 L1).
- **Post-booking** — {", ".join("`"+c+"`" for c in sorted(POST_BOOKING))}: known only at/after check-in
  (notebook 01 L2–L3).
- **Helper** — {", ".join("`"+c+"`" for c in sorted(HELPER))}: kept for ordering/reporting, not a model input.

## Metrics

- **Primary:** PR-AUC (average precision), recall (cancelled), F1 (cancelled).
- **Secondary:** ROC-AUC, precision, accuracy, Brier score, confusion matrix, calibration curve.
- Rationale: predicting "never cancels" scores ~{1 - y.mean():.1%} accuracy; R1's Random Forest
  (plan.md §16) scored 72.8% accuracy at 0.012 recall and is useless to a revenue manager.

### Reference floor (must be beaten on the primary metrics)

`DummyClassifier(strategy="prior")`, 5-fold CV on the training data:

```
{baseline_cv.round(4).to_string()}
```

## How a model notebook consumes this

**Model-tuning notebooks (`03_*`) load the training data only:**

```python
import json, pandas as pd
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
M = json.loads((ROOT / "data/processed/split_manifest.json").read_text())

train = pd.read_csv(ROOT / M["artifacts"]["train_csv"], index_col="row_id")   # training rows only

num  = M["feature_groups"]["numeric_scaled"]
ohe  = M["feature_groups"]["categorical_onehot"]
freq = M["feature_groups"]["categorical_frequency"]
for c in ohe + freq:
    train[c] = train[c].astype(str)

X_train, y_train = train[num + ohe + freq], train[M["target"]].astype(int)
```

Then copy `FrequencyEncoder`, `preprocess`, `make_pipeline`, `cv`, `SCORERS`, `evaluate`, `cv_report`
from notebook 02 (§5–§7), run one model's hyperparameter search inside `cv`, fit once on the full
training set, and stop.

**`test.csv` is never loaded in a model-tuning notebook.** It is read exactly once, in the Stage 3
scoring notebook, after the recommended model and its operating threshold are fixed:

```python
# Stage 3 only — after the model is chosen. Never in a 03_* tuning notebook.
test = pd.read_csv(ROOT / M["artifacts"]["test_csv"], index_col="row_id")
for c in ohe + freq:
    test[c] = test[c].astype(str)
X_test, y_test = test[num + ohe + freq], test[M["target"]].astype(int)
```
"""

out_path = REPORTS_DIR / "training_foundation.md"
out_path.write_text(foundation_md, encoding="utf-8")
print("wrote:", out_path.relative_to(ROOT), f"({len(foundation_md):,} characters)")

wrote: reports\training_foundation.md (5,119 characters)


---
## 13. Where this leaves us

**Done.** The cleaned table is split once (stratified, seeded, reproducible), the hold-out is written
and shut, the shared `ColumnTransformer` fits only inside training folds, the CV definition and the
metric harness are fixed, and `reports/training_foundation.md` tells each member how to plug in.

**Next — `notebooks/03_*.ipynb`, one per member, all on this foundation.** Logistic Regression
(baseline), Decision Tree, Random Forest, Gradient Boosting (XGBoost). Each: hyperparameter search
inside `cv`, one fit on the full training set, CV scores + confusion matrix + calibration +
importance/coefficients + a short error analysis. M1 assembles the comparison table; the single
held-out `test.csv` pass is the last step before the recommendation.